# Databricks Model Serving-- SKLearn example
In this notebook we start exploring model serving with a sample Linear Regression SKLearn model. This is an introductory notebook, where we explore how you can simply stitch together training a sample SKLearn model and then deploying to a Databricks Serving Endpoint.

### Environment
- <b>Provider</b>: Azure
- <b>Compute</b>: ML based instance, Runtime: 16.4LTS, Standard_DC4as_v5 (16GB memory, 4 cores)

### Additional Resources/Credits
- [Creating Serving Endpoints](https://docs.databricks.com/aws/en/machine-learning/model-serving/create-manage-serving-endpoints)
  - Reference SKLearn sample NB
- [Querying Endpoints](https://docs.databricks.com/aws/en/machine-learning/model-serving/score-custom-model-endpoints)


## Unity Catalog Setup
Register models to Unity Catalog, need to create a catalog and schema here, ensure you have your metastore location storage specified (Amazon: S3).

- Azure Metastore: https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/create-metastore#cloud-tenant-setup-azure
- AWS Metastore: https://docs.databricks.com/aws/en/data-governance/unity-catalog/create-metastore


Here we create our catalog and schema and specify the storage location (ensure metastore creation steps done above):

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS custom_ml
MANAGED LOCATION 'Enter managed storage location';

Validate creation:

In [0]:
%sql
SHOW CATALOGS;

Create schema:

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS custom_ml.models;

## MLFlow Setup
Here we create a sample SKLearn model and register it to Unity Catalog:

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Import mlflow
import mlflow
import mlflow.sklearn

In [0]:
# 1) Make mock regression data (simple + deterministic)
rng = np.random.default_rng(42)
n = 300

X = pd.DataFrame({
    "feature_1": rng.normal(loc=0.0, scale=1.0, size=n),
    "feature_2": rng.normal(loc=5.0, scale=2.0, size=n),
    "feature_3": rng.integers(low=0, high=10, size=n).astype(float),
})

# Linear-ish target with noise
y = 3.0 * X["feature_1"] - 1.2 * X["feature_2"] + 0.7 * X["feature_3"] + rng.normal(0, 0.5, n)
y = y.to_frame(name="target")  # keep as DataFrame like your example

# 2) Train/test split
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=42)

In [0]:
# 3) Unity Catalog target name, adjust for your use-case
catalog = "custom_ml"
schema = "models" 
# Adjust model name as needed
registered_name = f"{catalog}.{schema}.LinearRegressionMock"

# 4) Ensure we use Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")

In [0]:
# 5) Autolog + train inside an MLflow run
mlflow.sklearn.autolog(
    log_input_examples=True,
    registered_model_name=registered_name
)

# Can track this run in Experiments
with mlflow.start_run(run_name="lr_mock_train"):
    lr = LinearRegression()
    lr.fit(train_x, train_y)

    preds = lr.predict(test_x)

    # Simple metrics (optional but nice)
    rmse = mean_squared_error(test_y, preds, squared=False)
    r2 = r2_score(test_y, preds)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

print(f"Registered UC model: {registered_name}")

## Model Serving/Endpoint Creation
Here we can deploy our registered model, this streamlines MLOps operations for us (Model Registry -> Deployment). Note you can configure advanced options here as well like multiple models and Provisioned Concurrency, which we don't explore in this sample.

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client

mlflow.set_registry_uri("databricks-uc")
client = get_deploy_client("databricks") #Client to interfact with deployment & inference

# specify catalog, model name, version, etc
catalog = "custom_ml"
schema  = "models"
model_name = "LinearRegressionMock"   
version = "1" 

# Main Deployment API
endpoint = client.create_endpoint(
    name="sklearn-sample-ep",
    config={
        "served_entities": [
            {
                "name": f"{model_name}-{version}",                 # served model name on the endpoint
                "entity_name": f"{catalog}.{schema}.{model_name}", # UC model
                "entity_version": version,
                "workload_size": "Small",
                "workload_type": "CPU" #default
            }
        ]
    }
)

Takes a few minutes for creation, we poll the endpoint till it's in a ready status for inference.

In [0]:
import time
endpoint_name = "sklearn-sample-ep"

while True:
    ep = client.get_endpoint(endpoint_name)
    state = ep["state"]["ready"]

    print(f"Endpoint state: {state}")

    if state == "READY":
        print("✅ Endpoint is ready")
        break

    if state == "FAILED":
        raise RuntimeError(f"❌ Endpoint creation failed: {ep}")

    time.sleep(30)  # poll every 10s

## Sample Inference
Here we perform inference, you need to specify the column names and specific input values:

In [0]:
response = client.predict(
    endpoint="sklearn-sample-ep",
    inputs={
        "dataframe_split": {
            "index": [0],
            "columns": ["feature_1", "feature_2", "feature_3"],
            "data": [
                [0.15, 3.8, 7.0]
            ]
        }
    }
)
print(response)

In [0]:
%%time
for i in range(100):
    response = client.predict(
        endpoint="sklearn-sample-ep",
        inputs={
            "dataframe_split": {
                "index": [0],
                "columns": ["feature_1", "feature_2", "feature_3"],
                "data": [
                    [0.15, 4.8, 7.0]
                ]
            }
        }
    )